# Overlap-aware prediction: class-conditional conformal sets

About 80% of Analysis and Backdoor flows have an identical-feature twin of another class, so no model that sees these features can name their class reliably. Instead of forcing one label, this notebook returns a set of possible classes for every flow, built so that the true class is in the set with a stated probability, separately for every class (Mondrian conformal prediction).

Method, on the flows of UNSW-NB15_1.csv (637,891 after duplicate removal, the paper's data) and the same stratified 5:2:3 split as the GTCN-G style notebooks:
1. LightGBM (class-weighted or not) is trained on 85% of the training flows; the other 15% are used for early stopping only.
2. The validation flows (127,578) are the calibration set. For every class c, the nonconformity score of a calibration flow of class c is 1 minus the probability the model gives to c. The threshold for c is the ceil((n + 1)(1 - alpha))-th smallest of its n scores (infinite when n is too small, so the set then contains c for every flow).
3. A test flow's set holds every class whose probability is at least 1 minus its threshold, plus the most probable class so that no set is empty.
4. The test flows (191,368) are used once. Coverage is the share of flows of class c whose set contains c; the guarantee is coverage of at least 1 - alpha for every class, up to the sampling error of a finite test set.

Seeds change the model (training holdout and LightGBM seed); the split, calibration set and test set are fixed.

Two designs are tested. Design 1 puts a set over all 10 classes. Design 2 is two-stage: a binary attack detector whose threshold is set so that every attack class with enough calibration flows is detected with probability at least 0.95, then a set over the 9 attack classes only (a model trained on attack flows, calibrated on the validation attack flows that stage 1 flags, so that the guarantee holds for the flows the user actually sees). Worms has only 5 calibration flows, too few for any coverage guarantee, so it is left out of the guarantees: it enters a set only when it is the most probable class.

In [ ]:
import math

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split

PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
SEEDS = [42, 43, 44]
ALPHAS = [0.10, 0.05]
WEIGHTS = {"class weights": "balanced"}

## 1. Data and split

In [ ]:
df = pd.concat([pd.read_parquet(f"{PATH}/train.parquet"), pd.read_parquet(f"{PATH}/test.parquet")])
df = df[df.index < 700_001]
class_names = list(joblib.load(f"{PATH}/preprocessing.pkl")["label_encoder"].classes_)
n_classes = len(class_names)
feature_cols = [c for c in df.columns if c != "label"]

train_ids, rest_ids = train_test_split(df.index, train_size=0.5, stratify=df["label"], random_state=0)
val_ids, test_ids = train_test_split(rest_ids, train_size=0.4, stratify=df.loc[rest_ids, "label"], random_state=0)
X = {k: df.loc[ids, feature_cols] for k, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]}
y = {k: df.loc[ids, "label"].to_numpy() for k, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]}
print({k: len(v) for k, v in y.items()})
print("calibration flows per class:", dict(zip(class_names, np.bincount(y["val"], minlength=n_classes))))
print("test flows per class:", dict(zip(class_names, np.bincount(y["test"], minlength=n_classes))))

## 2. Design 1: sets over all 10 classes

In [ ]:
def thresholds(proba_cal, y_cal, alpha):
    q = np.full(n_classes, np.inf)
    for c in range(n_classes):
        scores = np.sort(1 - proba_cal[y_cal == c, c])
        k = math.ceil((len(scores) + 1) * (1 - alpha))
        if 0 < k <= len(scores):
            q[c] = scores[k - 1]
    return q


def prediction_sets(proba, q):
    sets = (1 - proba) <= q
    sets[np.arange(len(proba)), proba.argmax(1)] = True
    return sets


rows, composition, forced = [], [], []
for seed in SEEDS:
    fit_idx, stop_idx = train_test_split(np.arange(len(y["train"])), test_size=0.15, stratify=y["train"], random_state=seed)
    for wname, weight in WEIGHTS.items():
        model = lgb.LGBMClassifier(class_weight=weight, learning_rate=0.08, num_leaves=31, n_estimators=400, random_state=seed, n_jobs=4, verbose=-1)
        model.fit(X["train"].iloc[fit_idx], y["train"][fit_idx], eval_set=[(X["train"].iloc[stop_idx], y["train"][stop_idx])],
                  callbacks=[lgb.early_stopping(20, verbose=False)])
        p_cal, p_test = model.predict_proba(X["val"]), model.predict_proba(X["test"])
        print(f"seed {seed} | {wname} | trees {model.best_iteration_}", flush=True)
        pred = p_test.argmax(1)
        for c in range(n_classes):
            m = y["test"] == c
            forced.append({"weights": wname, "seed": seed, "class": class_names[c], "flows": int(m.sum()), "recall, one label": (pred[m] == c).mean()})
        for alpha in ALPHAS:
            sets = prediction_sets(p_test, thresholds(p_cal, y["val"], alpha))
            size = sets.sum(1)
            for c in range(n_classes):
                m = y["test"] == c
                rows.append({"weights": wname, "seed": seed, "alpha": alpha, "class": class_names[c], "flows": int(m.sum()),
                             "coverage": sets[m, c].mean(), "mean set size": size[m].mean(), "single-class share": (size[m] == 1).mean()})
                composition.append({"weights": wname, "seed": seed, "alpha": alpha, "true class": class_names[c],
                                    **{class_names[j]: sets[m, j].mean() for j in range(n_classes)}})
            rows.append({"weights": wname, "seed": seed, "alpha": alpha, "class": "all flows", "flows": len(size),
                         "coverage": sets[np.arange(len(size)), y["test"]].mean(), "mean set size": size.mean(), "single-class share": (size == 1).mean()})

results = pd.DataFrame(rows)
composition = pd.DataFrame(composition)
forced = pd.DataFrame(forced)
results.to_csv("/kaggle/working/conformal_runs.csv", index=False)
composition.to_csv("/kaggle/working/conformal_composition.csv", index=False)
forced.to_csv("/kaggle/working/conformal_one_label_recall.csv", index=False)

## 3. Results

Mean over 3 seeds. Coverage is the share of a class's test flows whose set contains the class; the target is 1 - alpha.

In [ ]:
summary = results.groupby(["weights", "alpha", "class"], sort=False)[["flows", "coverage", "mean set size", "single-class share"]].mean()
summary = summary.join(forced.groupby(["weights", "class"], sort=False)["recall, one label"].mean(), on=["weights", "class"])
summary.to_csv("/kaggle/working/conformal_summary.csv")
for (w, a), s in summary.groupby(level=["weights", "alpha"], sort=False):
    print(f"\n{w}, alpha {a} (target coverage {1 - a:.2f})")
    print(s.droplevel(["weights", "alpha"]).round(3).to_string())

In [ ]:
order = class_names + ["all flows"]
fig, axes = plt.subplots(1, 3, figsize=(20, 4.8))
data = results[results["weights"] == "class weights"]
sns.barplot(data=data, x="class", y="coverage", hue="alpha", order=order, errorbar="sd", palette="deep", ax=axes[0])
for a, color in zip(ALPHAS, ["C0", "C1"]):
    axes[0].axhline(1 - a, color=color, linestyle="--", linewidth=1)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Test coverage per class (class weights); dashed: targets")
sns.barplot(data=data, x="class", y="mean set size", hue="alpha", order=order, errorbar="sd", palette="deep", ax=axes[1])
axes[1].set_title("Mean set size for flows of each class")
sns.barplot(data=data, x="class", y="single-class share", hue="alpha", order=order, errorbar="sd", palette="deep", ax=axes[2])
axes[2].set_title("Share of flows whose set holds one class")
for ax in axes:
    ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.savefig("/kaggle/working/fig_01.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
comp = composition[(composition["weights"] == "class weights") & (composition["alpha"] == 0.10)].drop(columns=["weights", "seed", "alpha"])
comp = comp.groupby("true class", sort=False).mean().loc[class_names]
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(comp, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax)
ax.set_xlabel("class in the set")
ax.set_title("Share of flows of each true class whose set contains each class (class weights, alpha 0.10)")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_02.png", dpi=130, bbox_inches="tight")
plt.show()

## 4. Design 2: detect attacks, then a set over the attack classes

Stage 1 is a class-weighted binary LightGBM. Its threshold is the largest of the class-wise thresholds (alpha 0.05) over attack classes with enough calibration flows, so each of those classes is detected with probability at least 0.95. Stage 2 is a class-weighted 9-class LightGBM trained on attack flows only, with Mondrian conformal sets calibrated on the validation attack flows that stage 1 flags. A flow's end-to-end coverage is the share of flows of its class that are both flagged as attacks and have their class in the set. As a comparison, the share of detected flows whose true class is among the k most probable classes is reported for k = 1 to 4.

In [ ]:
normal = class_names.index("Normal")
attack_classes = [c for c in range(n_classes) if c != normal]
to_attack_index = {c: i for i, c in enumerate(attack_classes)}
ALPHA_DETECT = 0.05
ALPHAS_SET = [0.10, 0.05]


def set_thresholds(proba_cal, y_cal, alpha, labels):
    q = np.full(len(labels), -np.inf)
    for i in range(len(labels)):
        scores = np.sort(1 - proba_cal[y_cal == i, i])
        k = math.ceil((len(scores) + 1) * (1 - alpha))
        if 0 < k <= len(scores):
            q[i] = scores[k - 1]
    return q


rows2, composition2, trees2 = [], [], []
y_att = {k: (v != normal).astype(int) for k, v in y.items()}
for seed in SEEDS:
    fit_idx, stop_idx = train_test_split(np.arange(len(y["train"])), test_size=0.15, stratify=y["train"], random_state=seed)
    stage1 = lgb.LGBMClassifier(class_weight="balanced", learning_rate=0.08, num_leaves=31, n_estimators=400, random_state=seed, n_jobs=4, verbose=-1)
    stage1.fit(X["train"].iloc[fit_idx], y_att["train"][fit_idx], eval_set=[(X["train"].iloc[stop_idx], y_att["train"][stop_idx])],
               callbacks=[lgb.early_stopping(20, verbose=False)])
    p1_cal, p1_test = stage1.predict_proba(X["val"])[:, 1], stage1.predict_proba(X["test"])[:, 1]
    q_detect = []
    for c in attack_classes:
        scores = np.sort(1 - p1_cal[y["val"] == c])
        k = math.ceil((len(scores) + 1) * (1 - ALPHA_DETECT))
        if 0 < k <= len(scores):
            q_detect.append(scores[k - 1])
    flagged = (1 - p1_test) <= max(q_detect)

    att_train = np.where(y["train"] != normal)[0]
    y_train_att = np.array([to_attack_index[c] for c in y["train"][att_train]])
    fit2, stop2 = train_test_split(np.arange(len(att_train)), test_size=0.15, stratify=y_train_att, random_state=seed)
    stage2 = lgb.LGBMClassifier(class_weight="balanced", learning_rate=0.08, num_leaves=31, n_estimators=400, random_state=seed, n_jobs=4, verbose=-1)
    stage2.fit(X["train"].iloc[att_train[fit2]], y_train_att[fit2], eval_set=[(X["train"].iloc[att_train[stop2]], y_train_att[stop2])],
               callbacks=[lgb.early_stopping(20, verbose=False)])
    flagged_cal = (1 - p1_cal) <= max(q_detect)
    cal_att = np.where((y["val"] != normal) & flagged_cal)[0]
    y_cal_att = np.array([to_attack_index[c] for c in y["val"][cal_att]])
    p2_cal = stage2.predict_proba(X["val"].iloc[cal_att])
    p2_test = stage2.predict_proba(X["test"])
    print(f"seed {seed} | stage 1 trees {stage1.best_iteration_} | stage 2 trees {stage2.best_iteration_} | false alarm rate {flagged[y['test'] == normal].mean():.4f}", flush=True)
    trees2.append({"seed": seed, "stage 1 trees": stage1.best_iteration_, "stage 2 trees": stage2.best_iteration_})

    for alpha in ALPHAS_SET:
        q = set_thresholds(p2_cal, y_cal_att, alpha, attack_classes)
        sets = ((1 - p2_test) <= q)
        sets[np.arange(len(p2_test)), p2_test.argmax(1)] = True
        size = sets.sum(1)
        for c in attack_classes:
            m = y["test"] == c
            i = to_attack_index[c]
            rows2.append({"seed": seed, "alpha": alpha, "class": class_names[c], "flows": int(m.sum()), "detected": flagged[m].mean(),
                          "coverage, end to end": (flagged & sets[:, i])[m].mean(), "coverage, given detected": sets[m & flagged, i].mean() if (m & flagged).any() else np.nan,
                          "mean set size, detected flows": size[m & flagged].mean() if (m & flagged).any() else np.nan,
                          "single-class share, detected flows": (size[m & flagged] == 1).mean() if (m & flagged).any() else np.nan,
                          "calibration flows": int(((y["val"] == c) & flagged_cal).sum()),
                          **{f"coverage of the top {k} classes, given detected": (np.argsort(-p2_test, 1)[:, :k] == i).any(1)[m & flagged].mean() if (m & flagged).any() else np.nan for k in range(1, 5)}})
            composition2.append({"seed": seed, "alpha": alpha, "true class": class_names[c],
                                 **{class_names[attack_classes[j]]: sets[m & flagged, j].mean() if (m & flagged).any() else np.nan for j in range(len(attack_classes))}})
        rows2.append({"seed": seed, "alpha": alpha, "class": "Normal (false alarm rate)", "flows": int((y["test"] == normal).sum()), "detected": flagged[y["test"] == normal].mean()})

results2 = pd.DataFrame(rows2)
composition2 = pd.DataFrame(composition2)
results2.to_csv("/kaggle/working/conformal_two_stage_runs.csv", index=False)
composition2.to_csv("/kaggle/working/conformal_two_stage_composition.csv", index=False)
pd.DataFrame(trees2).to_csv("/kaggle/working/conformal_two_stage_trees.csv", index=False)

## 5. Design 2 results

Mean over 3 seeds. Detected is the share of the class's test flows flagged as attacks; coverage given detected is the share of detected flows whose set contains the class.

In [ ]:
cols = ["flows", "calibration flows", "detected", "coverage, end to end", "coverage, given detected", "mean set size, detected flows", "single-class share, detected flows"] + [f"coverage of the top {k} classes, given detected" for k in range(1, 5)]
summary2 = results2.groupby(["alpha", "class"], sort=False)[cols].mean()
summary2.to_csv("/kaggle/working/conformal_two_stage_summary.csv")
for a, s in summary2.groupby(level="alpha", sort=False):
    print(f"\nset alpha {a} (target coverage of the set {1 - a:.2f}, detection target {1 - ALPHA_DETECT:.2f})")
    print(s.droplevel("alpha").round(3).to_string())

In [ ]:
order = [class_names[c] for c in attack_classes]
data = results2[results2["class"].isin(order)]
fig, axes = plt.subplots(1, 3, figsize=(20, 4.8))
sns.barplot(data=data, x="class", y="coverage, end to end", hue="alpha", order=order, errorbar="sd", palette="deep", ax=axes[0])
axes[0].set_title("End-to-end coverage (flagged and class in set)")
axes[0].set_ylim(0, 1.05)
sns.barplot(data=data, x="class", y="mean set size, detected flows", hue="alpha", order=order, errorbar="sd", palette="deep", ax=axes[1])
axes[1].set_title("Mean set size over the 9 attack classes")
sns.barplot(data=data, x="class", y="detected", order=order, errorbar="sd", color="steelblue", ax=axes[2])
axes[2].set_title("Share of flows flagged as attacks")
for ax in axes:
    ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.savefig("/kaggle/working/fig_03.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
comp2 = composition2[composition2["alpha"] == 0.10].drop(columns=["seed", "alpha"]).groupby("true class", sort=False).mean().loc[order]
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(comp2, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax)
ax.set_xlabel("class in the set")
ax.set_title("Design 2: share of detected flows of each true class whose set contains each class (alpha 0.10)")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_04.png", dpi=130, bbox_inches="tight")
plt.show()